In [0]:
from pyspark.sql.functions import avg
df_customer = spark.table('samples.tpch.customer')
#muestra el saldo medio de los clientes por segmento de mercado:

df_segment_balance = df_customer.groupBy("c_mktsegment").agg(avg(df_customer["c_acctbal"]))
display(df_segment_balance)


In [0]:
df_segment_nation_balance = df_customer.groupBy("c_mktsegment", "c_nationkey").agg(avg(df_customer ["c_acctbal"]) )
display(df_segment_nation_balance)



In [0]:
#Para contar filas en un DataFrame, use el método
df_customer.count()

In [0]:
df_order=spark.table("samples.tpch.orders")
display(df_order)

In [0]:
# En el ejemplo siguiente se muestra cómo encadenar el filtrado, la agregación y el orden:
from pyspark.sql.functions import count, col

df_chained = ( df_order.filter(col("o_orderstatus") == "F")
              .groupBy(col("o_orderpriority"))
              .agg(count(col("o_orderkey")).alias("n_orders"))
              .sort(col("n_orders").desc())
              )
display(df_chained)

In [0]:
#guardar el DataFrame como una tabla en Unity Catalog
df_joined = df_order.join(
    df_customer, on = df_order["o_custkey"] == df_customer["c_custkey"],
    how = "inner"
)
display(df_joined)

In [0]:
##guardar el DataFrame como una tabla en Unity Catalog
catalog_name = "catalog_leo"
schema_name = "schema_leotienda"
table_name = "table_joined"

df_joined.write.saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")

In [0]:
# Assign this variable your file path
file_path = ""

(df_joined.write
  .format("csv")
  .mode("overwrite")
  .write(file_path)
)

In [0]:
df = spark.table('catalog_leo.schema_leotienda.table_joined')

In [0]:
df.count()

In [0]:
mostrar = df.select(df.c_phone.alias("celular")).collect()
display(mostrar)

In [0]:
#alias() and collect()
df_alias = spark.createDataFrame(
    [(2, "Alice"), (5,"Bob"), (25,"Jessy")], ["age","name"])
df_alias.select(df_alias.age.alias("edad")).collect()

In [0]:
#alias() and collect()
df_alias.select(df_alias.age.alias("super edad", metadata={'max': 99})).schema['super edad'].metadata['max']


In [0]:
from pyspark.sql import Row
df_asc = spark.createDataFrame([('Tom',80),('Alice',None),('Eli',3)], ["name","height"])
df_asc.select(df_asc.name).orderBy(df_asc.name.asc()).collect()



In [0]:
from pyspark.sql import Row
df_ascnull = spark.createDataFrame([('Tom',80),(None, 60),('Alice', None),(None,None)], ["name","height"])
df_ascnull.select(df_ascnull.name).orderBy(df_ascnull.name.asc_nulls_first()).collect()


In [0]:
from pyspark.sql import Row
df_asnot_nulo = spark.createDataFrame([('Tom',88),(None,60),('Alice',None),(None,None)],["Name", "Height"])
df_asnot_nulo.select(df_asnot_nulo.Name).orderBy(df_asnot_nulo.Name.asc_nulls_last()).collect()

In [0]:
df_entre = spark.createDataFrame([(2,"Alice"),(5,'Bob'),(34,'Jessy'),(21,'Eli')],["age","name"])
df_entre.select(df_entre.name, df_entre.age.between(2, 4).alias("edad limite")).show()

In [0]:
from pyspark.sql.functions import *

In [0]:
df_entre2 = spark.createDataFrame([("Alice","A"),('Bon','b'),('Jessy','z')],["name","initial"]) #Si en la columna initial hay minuscula no lo reconoce al ingresar mayuscula
df_entre2.select(df_entre2.name, upper(df_entre2.initial).between("A","B").alias("Entre")).show()

In [0]:
df_entre3 = spark.createDataFrame([(2.5,"Alice"),(5.5,"Bob"),(4.9,"Jessy"),(5.6,"Eli")],["height","name"])
df_entre3.select(df_entre3.height.between(2.0,5.0),df_entre3.name).show()

In [0]:
#withColumn
import pyspark.sql.functions as sf
df_entre4 = spark.createDataFrame([("Alice","2023-01-01"),("Bob","2023-02-01"),("Jessy","2023-03-03"),("Eli","2026-05-02")], ["name","date"])
df_entre4.show()

df_entre5 = df_entre4.withColumn("date",sf.to_date(df_entre4.date))

df_entre5.select(df_entre5.name,df_entre5.date.between("2023-01-01","2023-12-31")).show()

In [0]:
import pyspark.sql.functions as sf
df_entre5 = spark.createDataFrame([("Alice","2023-01-01 10:00:00"),("Bob","2023-02-01 10:00:00"),("Jessy"," 2026-09-26 11:00:59"),("Eli","2019-04-24 02:34:06"),],["name","timestamp"])
df_entre5 = df_entre5.withColumn("timestamp", sf.to_timestamp(df_entre5.timestamp))
df_entre5.select(df_entre5.name, df_entre5.timestamp.between("2019-01-01 00:00:00","2023-01-01 22:59:59").alias("Tiempo")).show()

In [0]:
import pyspark.sql.functions as sf
df_entre5 = spark.createDataFrame([("Alice","2023-01-01 10:00:00"),("Bob","2023-02-01 10:00:00"),("Jessy"," 2026-09-26 11:00:59"),(None,"2019-04-24 02:34:06"),],["name","timestamp"])
df_entre5 = df_entre5.withColumn("timestamp", sf.to_timestamp(df_entre5.timestamp))
df_entre5.select(df_entre5.name, df_entre5.timestamp.between("2019-01-01","2023-01-01").alias("Tiempo")).orderBy(df_entre5.name.asc_nulls_last()).show()

In [0]:
from pyspark.sql import Row
dfbitwi = spark.createDataFrame([Row(a=170,b=175,c=2,d=190)])
dfbitwi.select(dfbitwi.a.bitwiseAND(dfbitwi.c)).collect()

In [0]:
from pyspark.sql import Row
df_bitwi = spark.createDataFrame([Row(a=170,b=75,c=2,d=2)])
df_bitwi.select(df_bitwi.a.bitwiseOR(df_bitwi.c)).collect()

In [0]:
from pyspark.sql import Row
df_bitwi1 = spark.createDataFrame([Row(a=170,b=75,c=2,d=32)])
df_bitwi1.select(df_bitwi1.a.bitwiseXOR(df_bitwi1.b)).collect()

In [0]:
from pyspark.sql.types import StringType
#Para ver el cambio ejecutar primero el DF y luego funcion cast
df_cast = spark.createDataFrame([(2,"Alice"),(5,"Bob"),(29,"Jessy"),(21,"Eli")],["Age","Name"])
#df_cast=select(df_cast.Age.cast("string").alias('Ages')).collect()

In [0]:
from pyspark.sql.types import StringType
df_cast2 = spark.createDataFrame([(2,"Alice"),(5,"Bob"),(34,"Jessy"),(21,"Eli")],["age","name"])
df_cast2.select(df_cast2.age.cast(StringType()).alias('Ages')).collect()

In [0]:
df_contains = spark.createDataFrame(
     [(2, "Alice"), (5, "Bob"),(4,"oscar"),(6,"Mio")], ["age", "name"])
df_contains.filter(df_contains.name.contains('o')).collect()

In [0]:
from pyspark.sql import Row
df_desc = spark.createDataFrame([("Tom", 80),("Alice",None),("Maria",45),("Ronaldo",50)], ["name","height"])
df_desc.select(df_desc.name).orderBy(df_desc.name.desc()).collect()